# Week 1 Day 2: autonomy where it's needed, code where it isn't

Okafor Pascal's take on the Week 1 Day 2 assignment: with the agent approach, get at least 15 of 25 claims right *consistently* at the lowest cost, then push accuracy up cheaply.

Ed's agent spends model calls on steps that need no judgment (looking up a policy it could be handed, a closing message after the decision is recorded) and leaves the arithmetic to the model. This version moves those steps into code and gives the model tools only for the parts that need judgment:

1. **No fake autonomy.** The policy record is in the prompt, and recording the decision ends the run.
2. **Tools the agent chooses when it needs them.** `get_claim_history`, `search_claims_manual` and `read_manual_section`.
3. **Code does the arithmetic.** `calculate_payout` wraps Ed's `labs/week2day1_payout.py`. The model only classifies the lines, and `record_decision` refuses an approval until the payout has been calculated.
4. **The agent never sees `data/evals/`.** The labels are used only for scoring.

Every configuration runs 3 times. Results are cached in `results/`, so re-running this notebook costs nothing unless `RUN_PAID = True` and a run is missing.

## Setup

In [ ]:
import inspect
import json
import math
import os
import re
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, replace
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Literal

import httpx
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError

ROOT = Path("../..")
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "labs"))
# Ed's payout module, credited in the README
from week2day1_payout import Benefit, Line, work_out_payout  # noqa: E402

In [ ]:
load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key found" if openrouter_api_key else "No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

RUN_PAID = False  # Set True to make paid calls for runs that are not cached yet
BUDGET = 3.00  # US dollars across every cached run; run_config stops before going over it
WORKERS = 5  # Claims decided in parallel
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

## The data

The same loaders as the lab. `get_sample_data` is the only code that reads `data/evals/`, and what it returns goes only to the scoring functions, never into a prompt or a tool.

In [ ]:
def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


claims, labels = get_sample_data()
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
customers = {customer["customer_id"]: customer for customer in load("customers.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
claims_manual = (DATA / "claims_manual.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(policies)} policies, {len(customers)} customers")
print(f"Wording {len(policy_wording.split()):,} words, manual {len(claims_manual.split()):,} words")

## Ed's original agent

Reproduced from the lab for the baseline: the same instructions, prompt and two tools. The model asks for the policy, records its decision, then finishes with a message: three model calls a claim.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

ED_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def ed_claim_prompt(claim):
    """The claim alone, as JSON. Ed's agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


def ed_tools(state):
    """Ed's two tools, recording into this claim's state."""
    state["recorded"] = {}

    @tool
    def get_policy(policy_id: str) -> str:
        """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
        policy = policies.get(policy_id)
        return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
        """Record the decision on a claim in the claims system.

        payout is the amount to pay in US dollars, or 0 for a decline or a referral.
        rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
        """
        state["recorded"][claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
        return f"Recorded {decision} on {claim_id}"

    return [get_policy, record_decision]

## My tools

Every tool is built for one claim, so it knows the claim, the policy and the customer without the model passing ids around, and parallel claims can't mix up their state.

### The claims manual: generic BM25 search

The manual is split into its 14 numbered sections, and each section into paragraphs. `search_claims_manual` ranks every section with BM25 over plain word tokens and shows the best-matching paragraph of the top three. There's no keyword list and nothing tuned to the claims in the queue: the ranking only knows the manual's own words.

In [ ]:
WORD = re.compile(r"[a-z0-9]+")
SUFFIXES = ("ing", "ed", "es", "s")


def stem(word):
    """Strip one common English suffix, so that monkey matches monkeys and delay matches delayed."""
    for suffix in SUFFIXES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3 and not word.endswith("ss"):
            return word[: -len(suffix)]
    return word


def tokens(text):
    """Lower-case, lightly stemmed word tokens."""
    return [stem(word) for word in WORD.findall(text.lower())]


def split_manual(text):
    """Split the manual into its numbered sections: number, title, text and paragraphs."""
    sections = []
    for chunk in re.split(r"^## ", text, flags=re.MULTILINE)[1:]:
        heading, _, body = chunk.partition("\n")
        number, _, title = heading.partition(". ")
        paragraphs = [paragraph.strip() for paragraph in body.split("\n\n") if paragraph.strip()]
        sections.append({"number": int(number), "title": title.strip(), "text": body.strip(), "paragraphs": paragraphs})
    return sections


def bm25_scorer(documents, k1=1.5, b=0.75):
    """Return a function that scores a query against each document with BM25."""
    counts = [Counter(tokens(document)) for document in documents]
    lengths = [sum(count.values()) for count in counts]
    average = sum(lengths) / len(lengths)
    frequency = Counter(word for count in counts for word in count)
    idf = {word: math.log((len(documents) - df + 0.5) / (df + 0.5) + 1) for word, df in frequency.items()}

    def score(query):
        words = set(tokens(query))
        return [
            sum(
                idf[word] * count[word] * (k1 + 1) / (count[word] + k1 * (1 - b + b * length / average))
                for word in words
                if word in count
            )
            for count, length in zip(counts, lengths, strict=True)
        ]

    return score


MANUAL_SECTIONS = split_manual(claims_manual)
SECTION_SCORER = bm25_scorer([f"{section['title']} {section['text']}" for section in MANUAL_SECTIONS])


def search_manual(query, top=3):
    """Rank the manual's sections for a query, with the best-matching paragraph of each."""
    scores = SECTION_SCORER(query)
    ranked = sorted(zip(scores, MANUAL_SECTIONS, strict=True), key=lambda pair: -pair[0])[:top]
    results = []
    for score, section in ranked:
        if score <= 0:
            continue
        paragraph_scores = bm25_scorer(section["paragraphs"])(query)
        best = section["paragraphs"][paragraph_scores.index(max(paragraph_scores))]
        results.append({"section": section["number"], "title": section["title"], "score": round(score, 2), "best_paragraph": best})
    return results

### Claim history

The manual (section 9) refers a customer's third claim within any 12 months. The tool returns the earlier claims and does the date arithmetic in code. It returns only the claim history, none of the customer's contact details or date of birth (manual section 11).

In [ ]:
def claims_in_window(previous_claims, submitted_on, days=365):
    """Count the earlier claims filed in the 12 months up to this claim's submission date."""
    filed = date.fromisoformat(submitted_on)
    start = filed - timedelta(days=days)
    return sum(start < date.fromisoformat(earlier["filed_on"]) <= filed for earlier in previous_claims)


def claim_history(claim):
    """The customer's earlier claims and how many fall in the 12 months before this one."""
    customer = customers.get(claim["customer_id"])
    if customer is None:
        return {"error": f"There is no customer {claim['customer_id']}"}
    earlier = [previous for previous in customer["previous_claims"] if previous["claim_id"] != claim["claim_id"]]
    in_window = claims_in_window(earlier, claim["submitted_on"])
    return {
        "customer_since": customer["customer_since"],
        "earlier_claims": earlier,
        "earlier_claims_filed_in_the_12_months_before_this_one": in_window,
        "this_claim_is_number_in_those_12_months": in_window + 1,
    }

### Payout: Ed's module does the arithmetic

`calculate_payout` passes the model's lines to `work_out_payout` from `labs/week2day1_payout.py`. Its `Line` schema already explains each kind of line (item, electronics, cash, expense, hotel, meals without receipts), so the model's job is to classify, not to add up.

In [ ]:
def payout_for(policy, benefit, lines, paid_by_others=0.0):
    """Work out the payout with Ed's module, returning the calculation or an error the model can act on."""
    try:
        return work_out_payout(policy, benefit, [Line.model_validate(line) for line in lines], paid_by_others)
    except (KeyError, ValueError, TypeError) as error:
        return {"error": f"Could not work out the payout: {error}"}

### The tools for one claim

`record_decision` asks for the rationale *before* the decision (this is v2; see below). The lab makes the same point about Structured Outputs: the model should reason before it commits.

`record_decision` stops the run once a decision is recorded. `return_direct=True` works in this LangChain version (1.4), but it ends the run after *any* call to the tool, including a refused one, which would leave the claim with nothing recorded. So the stop is a `before_model` middleware instead: as soon as a decision is on file, the agent ends without another model call.

With the payout tool on, `record_decision` refuses `approve` until `calculate_payout` has run on this claim, and records the computed payout rather than a number the model types.

The same middleware gives the agent a step budget of 10 model calls. An agent that runs out ends cleanly with nothing recorded, which counts as wrong, and the run keeps the list of tools it called so the loop can be seen.

In [ ]:
Decision = Literal["approve", "decline", "refer"]


def my_tools(claim, state, manual_and_history, payout_tool):
    """Build the tools for one claim, according to which levers are on."""
    policy = policies[claim["policy_id"]]
    tools = []

    if payout_tool:

        @tool
        def calculate_payout(benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> str:
            """Work out the payout on an approved claim with UnderCover's payout rules (policy wording section 10).

            List each covered item or expense as a line and leave out any excluded ones. The code applies the
            per-item and other limits, takes off what others paid and the deductible, and caps at the benefit
            maximum. paid_by_others is anything a carrier, hotel or other party has paid or refunded for the same loss.
            """
            result = payout_for(policy, benefit, [line.model_dump() for line in lines], paid_by_others)
            if "payout" in result:
                state["calculated"] = result["payout"]
            return json.dumps(result, indent=2)

        @tool
        def record_decision(rationale: str, decision: Decision) -> str:
            """Record the decision on this claim. Recording it ends your work on the claim.

            Write the rationale first: 2-3 sentences that work through the claim and name the relevant section of the
            wording or manual. Then give the decision that follows from it.
            For approve, call calculate_payout first: the payout it worked out is the one recorded.
            """
            if decision == "approve" and "calculated" not in state:
                return "Not recorded: call calculate_payout before approving, then record the decision again."
            payout = state["calculated"] if decision == "approve" else 0.0
            state["answer"] = {"decision": decision, "payout": payout, "rationale": rationale}
            return f"Recorded {decision} with a payout of ${payout:,.2f}"

        tools += [calculate_payout, record_decision]
    else:

        @tool
        def record_decision(rationale: str, decision: Decision, payout: float) -> str:
            """Record the decision on this claim. Recording it ends your work on the claim.

            Write the rationale first: 2-3 sentences that work through the claim and name the relevant section of the
            wording or manual. Then give the decision that follows from it.
            payout is the amount to pay in US dollars, or 0 for a decline or a referral.
            """
            state["answer"] = {"decision": decision, "payout": payout, "rationale": rationale}
            return f"Recorded {decision} with a payout of ${payout:,.2f}"

        tools.append(record_decision)

    if manual_and_history:

        @tool
        def get_claim_history() -> str:
            """Look up this customer's earlier claims, with how many were filed in the 12 months before this one."""
            return json.dumps(claim_history(claim), indent=2)

        @tool
        def search_claims_manual(query: str) -> str:
            """Search UnderCover's internal claims handling manual. Returns the best-matching sections, each with its most relevant paragraph."""
            results = search_manual(query)
            return json.dumps(results, indent=2) if results else "No section of the manual matches that query."

        @tool
        def read_manual_section(section: int) -> str:
            """Read one numbered section of the claims manual in full."""
            for candidate in MANUAL_SECTIONS:
                if candidate["number"] == section:
                    return f"## {candidate['number']}. {candidate['title']}\n\n{candidate['text']}"
            return f"There is no section {section}. The manual has sections 1 to {len(MANUAL_SECTIONS)}."

        tools += [get_claim_history, search_claims_manual, read_manual_section]

    return tools


MAX_MODEL_CALLS = 10  # The full agent needs about 4; more than 10 means it is going round in circles


def stop_run(state, max_calls=MAX_MODEL_CALLS):
    """Middleware that ends the run with no further model call once a decision is recorded, or once the step budget is spent."""

    @before_model(can_jump_to=["end"])
    def stop(agent_state, runtime):
        if "answer" in state:
            return {"jump_to": "end"}
        state["model_calls"] = state.get("model_calls", 0) + 1
        if state["model_calls"] > max_calls:
            state["stopped"] = f"step budget of {max_calls} model calls"
            return {"jump_to": "end"}
        return None

    return stop

### My prompts

The same instructions as Ed's, plus one short paragraph for each lever that is on. The guidance describes what each tool is for. It doesn't mention any claim in the queue.

In [ ]:
TOOL_GUIDANCE = {
    "base": "The customer's policy record is in the message. Record your decision with record_decision; recording it ends your work on the claim.",
    "manual_and_history": (
        "UnderCover's claims manual holds internal handling rules for what the wording leaves open. "
        "Use search_claims_manual and read_manual_section when the wording alone does not settle the claim. "
        "Use get_claim_history to see the customer's earlier claims."
    ),
    "payout_tool": (
        "To approve, list the covered lines and call calculate_payout, which does the arithmetic; "
        "record_decision then records the payout it worked out. Do not add up the payout yourself."
    ),
}


def my_system_prompt(manual_and_history, payout_tool):
    """Ed's instructions and the wording, with the guidance for the levers that are on."""
    instructions = INSTRUCTIONS
    guidance = [TOOL_GUIDANCE["base"]]
    if manual_and_history:
        guidance.append(TOOL_GUIDANCE["manual_and_history"])
    if payout_tool:
        instructions = instructions.replace("Work out the payout using section 10.", "Work out the payout with calculate_payout.")
        guidance.append(TOOL_GUIDANCE["payout_tool"])
    guidance_text = "\n".join(guidance)
    return f"""{instructions}
{guidance_text}

{policy_wording}
"""


def my_claim_prompt(claim):
    """The customer's policy record and the claim, as JSON, as in the lab's single-call prompt."""
    return f"""# Policy record

{json.dumps(policies[claim["policy_id"]], indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

## Running one claim

A configuration names the model and which levers are on. A callback adds up OpenRouter's cost and tokens from every model call and lists every tool call as it starts, so a run that fails or loops still shows what it spent and what it did.

`version` 1 is the first ablation, whose `record_decision` asked for the decision before the rationale. Its runs are cached and are loaded, never re-run: the code that made them is at commit `f913338`.

In [ ]:
@dataclass(frozen=True)
class Config:
    """One agent configuration."""

    name: str
    model: str
    effort: str | None = "low"
    ed_original: bool = False
    manual_and_history: bool = False
    payout_tool: bool = False
    version: int = 2
    requests_per_minute: float | None = None  # Pace the model calls, for free models' per-minute limit
    rate_limit_retries: int = 0  # Retry a claim this many times after a 429, waiting 30s, 60s, 120s, ...


class RunTracker(BaseCallbackHandler):
    """Adds up OpenRouter's cost, tokens and model calls, and lists the tools called."""

    def __init__(self):
        self.cost = 0.0
        self.calls = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self.tools = []

    def on_llm_end(self, response, **kwargs):
        for generations in response.generations:
            for generation in generations:
                message = getattr(generation, "message", None)
                usage = message.response_metadata.get("token_usage", {}) if message else {}
                self.cost += usage.get("cost") or 0.0
                self.input_tokens += usage.get("prompt_tokens") or 0
                self.output_tokens += usage.get("completion_tokens") or 0
                self.calls += 1

    def on_tool_start(self, serialized, input_str, **kwargs):
        self.tools.append((serialized or {}).get("name") or kwargs.get("name") or "unknown")


RATE_LIMITERS = {}


def rate_limiter_for(config):
    """One shared limiter for each paced model, so its calls stay under the per-minute limit."""
    if not config.requests_per_minute:
        return None
    key = (config.model, config.requests_per_minute)
    if key not in RATE_LIMITERS:
        RATE_LIMITERS[key] = InMemoryRateLimiter(requests_per_second=config.requests_per_minute / 60, check_every_n_seconds=0.5, max_bucket_size=1)
    return RATE_LIMITERS[key]


def make_llm(config):
    """The model through OpenRouter, asking only for providers that support every parameter we send."""
    options = {"reasoning_effort": config.effort} if config.effort else {}
    if limiter := rate_limiter_for(config):
        options["rate_limiter"] = limiter
    return ChatOpenAI(
        model=config.model,
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        timeout=120,
        max_retries=5,
        extra_body={"provider": {"require_parameters": True}},
        **options,
    )


def build_agent(config, claim, state):
    """The agent for one claim and its first message."""
    if config.ed_original:
        agent = create_agent(model=make_llm(config), tools=ed_tools(state), system_prompt=ED_SYSTEM_PROMPT)
        return agent, ed_claim_prompt(claim)
    tools = my_tools(claim, state, config.manual_and_history, config.payout_tool)
    agent = create_agent(
        model=make_llm(config),
        tools=tools,
        system_prompt=my_system_prompt(config.manual_and_history, config.payout_tool),
        middleware=[stop_run(state)],
    )
    return agent, my_claim_prompt(claim)


NOTHING = {"decision": "nothing recorded", "payout": 0.0, "rationale": ""}


def is_rate_limit(error):
    """True for OpenRouter's 429 rate-limit error."""
    return "429" in error or "RateLimit" in error


def decide(config, claim):
    """Run the agent on one claim. Returns what it recorded, the cost, tokens, calls and time, the tools it used and why it stopped.

    With rate_limit_retries, a claim that hits a 429 starts again after a wait that doubles each time.
    """
    start = time.time()
    for attempt in range(config.rate_limit_retries + 1):
        state = {}
        tracker = RunTracker()
        error, stopped = "", ""
        try:
            agent, prompt = build_agent(config, claim, state)
            agent.invoke({"messages": [{"role": "user", "content": prompt}]}, {"callbacks": [tracker], "recursion_limit": 50})
        except GraphRecursionError:  # Ed's agent has no step budget of its own, so LangGraph's limit is its backstop
            stopped = "LangGraph recursion limit"
        except Exception as exc:  # A failed claim counts as wrong; the error is kept for the misses table
            error = f"{type(exc).__name__}: {exc}"[:300]
        if not (is_rate_limit(error) and attempt < config.rate_limit_retries):
            break
        wait = 30 * 2**attempt
        print(f"{config.name} {claim['claim_id']}: rate-limited, waiting {wait}s before retry {attempt + 1} of {config.rate_limit_retries}")
        time.sleep(wait)
    answer = state.get("recorded", {}).get(claim["claim_id"], NOTHING) if config.ed_original else state.get("answer", NOTHING)
    return {
        "claim_id": claim["claim_id"],
        **answer,
        "cost": tracker.cost,
        "calls": tracker.calls,
        "input_tokens": tracker.input_tokens,
        "output_tokens": tracker.output_tokens,
        "seconds": round(time.time() - start, 1),
        "tools_used": tracker.tools,
        "stopped": stopped or state.get("stopped", ""),
        "error": error,
    }

## Scoring and caching

`is_right` is the lab's: the decision must match and the payout must be within a dollar. A miss is either a **wrong decision**, or the right decision with a **wrong payout**.

Each run is saved to `results/<set>/<config>__run<n>.json`. `run_config` loads a cached run if there is one. It makes paid calls only when `RUN_PAID` is True, and never when the cached spend has reached `BUDGET`.

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def score(rows, labels):
    """Add the label's difficulty, whether the answer is right, and the kind of miss."""
    scored = []
    for row in rows:
        label = labels[row["claim_id"]]
        right = is_right(row, label)
        miss = "" if right else ("wrong payout" if row["decision"] == label["decision"] else "wrong decision")
        scored.append({**row, "difficulty": label["difficulty"], "right": right, "miss": miss})
    return scored


def spent():
    """Total cost of every cached run, the pilot included."""
    return sum(row["cost"] for path in RESULTS.rglob("*.json") if path.name != "models.json" for row in json.loads(path.read_text())["rows"])


class RunAborted(Exception):
    """A paced run stopped before the end: still rate-limited after its retries, or past its deadline."""


def run_paced(config, claims, path, deadline):
    """Decide the claims one at a time, stopping cleanly at the deadline or on a claim that stays rate-limited."""
    rows = []
    for claim in claims:
        reason = ""
        if deadline and datetime.now() >= deadline:
            reason = f"stopped at the {deadline:%H:%M} deadline"
        else:
            rows.append(decide(config, claim))
            if is_rate_limit(rows[-1]["error"]):
                reason = f"still rate-limited after {config.rate_limit_retries} retries on {claim['claim_id']}"
        if reason:
            partial = path.with_name(path.stem + ".aborted.json")
            partial.write_text(json.dumps({"config": asdict(config), "reason": reason, "rows": rows}, indent=2))
            raise RunAborted(f"{config.name}: {reason}, after {len(rows)} of {len(claims)} claims")
    return rows


def run_config(config, run, claims, labels, set_name="queue", workers=WORKERS, deadline=None):
    """Run a configuration once over a set of claims, or load the cached run.

    A paced configuration (requests_per_minute set) runs one claim at a time and can stop early with RunAborted.
    """
    path = RESULTS / set_name / f"{config.name}__run{run}.json"
    if path.exists():
        return score(json.loads(path.read_text())["rows"], labels)
    if config.version == 1:
        raise RuntimeError(f"{path} is a v1 run, which this code can't reproduce. Check out commit f913338 to re-run it.")
    if not RUN_PAID:
        raise RuntimeError(f"{path} is not cached. Set RUN_PAID = True to run it.")
    if spent() >= BUDGET:
        raise RuntimeError(f"The cached runs have spent ${spent():.2f}, at or over the ${BUDGET:.2f} budget.")
    path.parent.mkdir(parents=True, exist_ok=True)
    if config.requests_per_minute:
        rows = run_paced(config, claims, path, deadline)
    else:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            rows = list(pool.map(lambda claim: decide(config, claim), claims))
    path.write_text(json.dumps({"config": asdict(config), "run": run, "rows": rows}, indent=2))
    scored = score(rows, labels)
    cost = sum(row["cost"] for row in rows)
    print(f"{config.name} run {run}: {sum(row['right'] for row in scored)} of {len(rows)} right for ${cost:.4f} (total spent ${spent():.2f})")
    return scored


def run_repeats(config, claims, labels, runs=3, set_name="queue", workers=WORKERS, cutoff=0, deadline=None):
    """Run a configuration several times. Returns one list of scored rows for each run.

    With a cutoff, a configuration whose first run gets fewer than cutoff claims right stops after that run.
    """
    first = run_config(config, 1, claims, labels, set_name, workers, deadline)
    if sum(row["right"] for row in first) < cutoff:
        return [first]
    return [first] + [run_config(config, run, claims, labels, set_name, workers, deadline) for run in range(2, runs + 1)]


def cached_runs(config, labels, set_name="queue", runs=3):
    """The complete runs of a configuration that are already cached."""
    paths = [RESULTS / set_name / f"{config.name}__run{run}.json" for run in range(1, runs + 1)]
    return [score(json.loads(path.read_text())["rows"], labels) for path in paths if path.exists()]

## Summaries

For each configuration: min, mean and max correct over its runs, cost per run, cost per correct claim, seconds and model calls per claim, and the misses by difficulty and by kind.

In [ ]:
DIFFICULTIES = ["routine", "hard", "manual"]


def summarize(name, runs):
    """One row of results for a configuration's runs."""
    rows = [row for run in runs for row in run]
    correct = [sum(row["right"] for row in run) for run in runs]
    cost = sum(row["cost"] for row in rows)
    misses = [row for row in rows if not row["right"]]
    summary = {
        "configuration": name,
        "runs": len(runs),
        "min": min(correct),
        "mean": round(sum(correct) / len(runs), 1),
        "max": max(correct),
        "cost per run": cost / len(runs),
        "cost per correct": cost / max(sum(correct), 1),
        "seconds per claim": sum(row["seconds"] for row in rows) / len(rows),
        "calls per claim": sum(row["calls"] for row in rows) / len(rows),
    }
    for difficulty in DIFFICULTIES:
        summary[f"{difficulty} misses"] = round(sum(row["difficulty"] == difficulty for row in misses) / len(runs), 1)
    summary["wrong decision"] = round(sum(row["miss"] == "wrong decision" for row in misses) / len(runs), 1)
    summary["wrong payout"] = round(sum(row["miss"] == "wrong payout" for row in misses) / len(runs), 1)
    summary["stopped or failed"] = sum(bool(row["error"] or row.get("stopped")) for row in rows)
    return summary


def results_table(experiments):
    """A table of summaries, from a dict of configuration name to its runs."""
    table = pd.DataFrame([summarize(name, runs) for name, runs in experiments.items()]).set_index("configuration")
    return table.style.format({
        "cost per run": "${:.4f}",
        "cost per correct": "${:.5f}",
        "seconds per claim": "{:.1f}",
        "calls per claim": "{:.2f}",
    })


def stop_note(row):
    """Why a run ended without a normal decision, with the tools it called most."""
    reason = row.get("stopped") or row["error"][:60]
    if not reason:
        return ""
    looped = ", ".join(f"{name} x{count}" for name, count in Counter(row.get("tools_used", [])).most_common(3))
    return f" ({reason}; tools: {looped or 'none'})"


def show_misses(runs, labels):
    """Each claim missed in any run, how often, and what the agent and the handler said."""
    by_claim = {}
    for run in runs:
        for row in run:
            if not row["right"]:
                by_claim.setdefault(row["claim_id"], []).append(row)
    for claim_id, rows in sorted(by_claim.items(), key=lambda item: DIFFICULTIES.index(labels[item[0]]["difficulty"])):
        label = labels[claim_id]
        answers = ", ".join(f"{row['decision']} ${row['payout']:,.2f}" + stop_note(row) for row in rows)
        display(Markdown(f"""**{claim_id}, {label['difficulty']}**, missed {len(rows)} of {len(runs)}: {answers}; the handler said {label['decision']} ${label['payout']:,.2f}

- Agent: {rows[0]['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))

## Offline checks

These run with no model calls. They check the payout tool against the manual's worked examples, the 12-month count, the manual search, and that nothing the agent sees can reach `data/evals/`.

In [ ]:
economy = {"plan": "economy", "add_ons": [], "travelers": ["A"], "trip": {"trip_cost": 2000.0}}
premium_electronics = {"plan": "premium", "add_ons": ["electronics_plus"], "travelers": ["A"], "trip": {"trip_cost": 2000.0}}
first = {"plan": "first", "add_ons": [], "travelers": ["A"], "trip": {"trip_cost": 4000.0}}

# Worked example 1: jacket over the Economy per-item limit, less the $100 deductible
example_1 = payout_for(economy, "baggage", [
    {"description": "Jacket", "amount": 220, "kind": "item"},
    {"description": "Boots", "amount": 140, "kind": "item"},
    {"description": "Book", "amount": 25, "kind": "item"},
])
assert example_1["payout"] == 215.0, example_1

# Worked example 2: a trip delay capped at the First maximum, needing sign-off
example_2 = payout_for(first, "trip_delay", [{"description": "Hotel and meals", "amount": 1720, "kind": "expense"}])
assert example_2["payout"] == 1500.0 and example_2["needs_sign_off"], example_2

# Worked example 3: a laptop at the Electronics Plus limit, less the $50 Premium deductible
example_3 = payout_for(premium_electronics, "baggage", [{"description": "Laptop", "amount": 1450, "kind": "electronics"}])
assert example_3["payout"] == 1150.0 and example_3["needs_sign_off"], example_3

# A bad line comes back as an error the model can read, not an exception
assert "error" in payout_for(economy, "baggage", [{"description": "Mystery", "amount": 10, "kind": "spaceship"}])

# The 12-month window counts claims filed in the year up to the new claim, and not the ones before it
history = [{"filed_on": "2025-09-01"}, {"filed_on": "2026-03-10"}, {"filed_on": "2024-01-05"}]
assert claims_in_window(history, "2026-07-01") == 2
assert claims_in_window(history, "2027-03-11") == 0

# The generic search finds the right section for questions phrased in a customer's words
assert stem("monkeys") == stem("monkey") and stem("delayed") == stem("delay")
assert search_manual("a monkey grabbed my sunglasses")[0]["section"] == 7
assert search_manual("third claim in twelve months")[0]["section"] == 9
assert search_manual("meals at the airport without receipts")[0]["section"] == 10
assert len(MANUAL_SECTIONS) == 14

# Nothing the agent sees can reach the evaluation set
agent_facing = [ed_tools, my_tools, claim_history, search_manual, my_system_prompt, my_claim_prompt, ed_claim_prompt, build_agent]
for function in agent_facing:
    source = inspect.getsource(function)
    assert "evals" not in source and "labels" not in source, function.__name__
assert "evals" not in my_system_prompt(True, True) and "evals" not in ED_SYSTEM_PROMPT

print("All offline checks pass")

## Configurations

The baseline is Ed's agent with Luna at `low`. The ablation adds one lever at a time on the same model:

| Configuration | Policy in prompt, stop on record | Manual and history tools | Payout tool |
|---|---|---|---|
| `ed_original` | | | |
| `A_lean` | ✓ | | |
| `B_manual_history` | ✓ | ✓ | |
| `C_full` | ✓ | ✓ | ✓ |

There are two versions of A, B and C. **v1** asked for the decision before the rationale in `record_decision`. **v2** asks for the rationale first and adds the step budget. v1's runs are kept as they were.

In [ ]:
LUNA = "openai/gpt-6-luna"

ED_ORIGINAL = Config("ed_original", LUNA, ed_original=True)
ABLATION_V1 = [
    ED_ORIGINAL,
    Config("A_lean", LUNA, version=1),
    Config("B_manual_history", LUNA, manual_and_history=True, version=1),
    Config("C_full", LUNA, manual_and_history=True, payout_tool=True, version=1),
]
ABLATION_V2 = [
    Config("A_lean_v2", LUNA),
    Config("B_manual_history_v2", LUNA, manual_and_history=True),
    Config("C_full_v2", LUNA, manual_and_history=True, payout_tool=True),
]

# The full agent's tools, called directly with no model: approve is refused until the payout is calculated
state = {}
full_tools = {built.name: built for built in my_tools(claims[0], state, manual_and_history=True, payout_tool=True)}
assert full_tools["record_decision"].invoke({"rationale": "Covered.", "decision": "approve"}).startswith("Not recorded")
assert "answer" not in state
full_tools["calculate_payout"].invoke({"benefit": "trip_delay", "lines": [{"description": "Hotel", "amount": 245, "kind": "expense"}]})
full_tools["record_decision"].invoke({"rationale": "Covered.", "decision": "approve"})
assert state["answer"]["payout"] == 245.0
assert stop_run(state).before_model(None, None) == {"jump_to": "end"}

# The rationale comes first in both versions of record_decision
for payout_tool in (False, True):
    recorder = {built.name: built for built in my_tools(claims[0], {}, False, payout_tool)}["record_decision"]
    assert next(iter(recorder.args)) == "rationale"

# The step budget ends the run cleanly once the agent has used its model calls without recording anything
looping = {}
budget = stop_run(looping, max_calls=3)
assert [budget.before_model(None, None) for _ in range(4)] == [None, None, None, {"jump_to": "end"}]
assert looping["stopped"] == "step budget of 3 model calls"

# Every configuration builds its agent without calling a model
for config in ABLATION_V1 + ABLATION_V2:
    build_agent(config, claims[0], {})
print("The tools and every configuration check out")

## Pilot: the cost of one claim

Before any full run, each configuration decides two claims: CLM-1001 (a routine trip delay) and CLM-1019 (one that needs the manual and the history). The cost per claim gives the estimate for the full runs. This pilot ran with v1.

In [ ]:
PILOT_IDS = ["CLM-1001", "CLM-1019"]
pilot_claims = [claim for claim in claims if claim["claim_id"] in PILOT_IDS]

pilot = {config.name: run_config(config, 1, pilot_claims, labels, set_name="pilot") for config in ABLATION_V1}
for name, rows in pilot.items():
    for row in rows:
        print(f"{name:18} {row['claim_id']} {row['decision']:16} ${row['payout']:>8,.2f} right={row['right']!s:5} "
              f"${row['cost']:.5f} {row['calls']} calls {row['seconds']}s tools={row['tools_used']}")

In [ ]:
pilot_per_claim = {name: sum(row["cost"] for row in rows) / len(rows) for name, rows in pilot.items()}
estimate = pd.DataFrame({
    "cost per claim": pilot_per_claim,
    "one run of 25": {name: cost * 25 for name, cost in pilot_per_claim.items()},
    "three runs": {name: cost * 75 for name, cost in pilot_per_claim.items()},
})
print(f"Estimated baseline and ablation, 3 runs each: ${estimate['three runs'].sum():.2f}")
estimate.style.format("${:.4f}")

## Ablation v1

Each configuration runs 3 times over the 25 claims, on Luna at `low`.

In [ ]:
ablation_v1 = {config.name: run_repeats(config, claims, labels) for config in ABLATION_V1}
results_table(ablation_v1)

### What each lever added in v1

Each row adds one lever to the row above it, so the change columns show what that lever alone added.

In [ ]:
LEVERS = ["(Ed's agent)", "policy in prompt, stop on record", "manual and history tools", "payout tool"]


def lever_table(experiments):
    """What each lever added to the configuration above it: correct, cost and calls."""
    summary = pd.DataFrame([summarize(name, runs) for name, runs in experiments.items()]).set_index("configuration")
    table = pd.DataFrame({
        "lever added": LEVERS[: len(summary)],
        "mean correct": summary["mean"],
        "change in correct": summary["mean"].diff().round(1),
        "cost per run": summary["cost per run"],
        "change in cost": summary["cost per run"].diff(),
        "calls per claim": summary["calls per claim"],
    })
    return table.style.format({"cost per run": "${:.4f}", "change in cost": "${:+.4f}", "calls per claim": "{:.2f}"}, na_rep="")


def miss_table(experiments):
    """The mean misses in a run, by the claim's difficulty and by kind."""
    summary = pd.DataFrame([summarize(name, runs) for name, runs in experiments.items()]).set_index("configuration")
    return summary[[f"{difficulty} misses" for difficulty in DIFFICULTIES] + ["wrong decision", "wrong payout", "stopped or failed"]]


lever_table(ablation_v1)

Taking away the fake autonomy cut the cost and the calls but lost 4.4 claims a run. The cause is in the rationales. On CLM-1013, `A_lean` wrote *"exceeding the Economy plan's 12-hour Trip Delay waiting period? Actually it does not, so the claim is not eligible"* and recorded **approve $64**. v1's `record_decision` asked for the decision and payout before the rationale, so with a single call the model committed first and reasoned afterwards. In Ed's agent the `get_policy` turn came before the decision, and it was acting as thinking time.

In [ ]:
miss_table(ablation_v1)

## Ablation v2: rationale first, and a step budget

The same three levers, with `record_decision` asking for the rationale before the decision, and the 10-call step budget. Ed's agent is unchanged, so its v1 runs serve as the baseline here too.

In [ ]:
ablation_v2 = {ED_ORIGINAL.name: ablation_v1[ED_ORIGINAL.name]} | {config.name: run_repeats(config, claims, labels) for config in ABLATION_V2}
results_table(ablation_v2)

In [ ]:
lever_table(ablation_v2)

In [ ]:
miss_table(ablation_v2)

### Claim by claim

How many of its 3 runs each configuration got right on each claim, v1 and v2 side by side.

In [ ]:
def right_by_claim(experiments, labels):
    """A claim-by-configuration table of how many runs got each claim right."""
    table = pd.DataFrame({
        name: Counter(row["claim_id"] for run in runs for row in run if row["right"])
        for name, runs in experiments.items()
    }).reindex(list(labels)).fillna(0).astype(int)
    table.insert(0, "difficulty", [labels[claim_id]["difficulty"] for claim_id in table.index])
    return table.sort_values("difficulty", key=lambda column: column.map(DIFFICULTIES.index), kind="stable")


right_by_claim(ablation_v1 | ablation_v2, labels)

The v2 full agent's misses:

In [ ]:
show_misses(ablation_v2["C_full_v2"], labels)

## Model sweep

OpenRouter's model list is fetched once and cached in `results/models.json`. The sweep keeps the models that support tools and ranks the paid ones by an estimated cost per claim, using the full agent's measured tokens per claim on Luna. It leaves out `:batch` and `~` alias duplicates and takes the five cheapest, plus three free models.

Every model runs the v2 full agent. Each is piloted on the same two claims first. A model that gets fewer than 10 right on its first full run stops there.

In [ ]:
def fetch_models():
    """OpenRouter's model list, cached, keeping only the fields the sweep uses."""
    path = RESULTS / "models.json"
    if not path.exists():
        response = httpx.get(f"{OPENROUTER_BASE_URL}/models", timeout=30)
        response.raise_for_status()
        keep = [
            {
                "id": model["id"],
                "prompt": float(model["pricing"].get("prompt") or 0),
                "completion": float(model["pricing"].get("completion") or 0),
                "context_length": model.get("context_length"),
                "supported_parameters": model.get("supported_parameters") or [],
            }
            for model in response.json()["data"]
        ]
        path.write_text(json.dumps(keep, indent=1))
    return json.loads(path.read_text())


luna_rows = [row for run in ablation_v2["C_full_v2"] for row in run]
TOKENS_IN = sum(row["input_tokens"] for row in luna_rows) / len(luna_rows)
TOKENS_OUT = sum(row["output_tokens"] for row in luna_rows) / len(luna_rows)


def per_claim(model):
    """Estimated cost of one claim for the full agent, at list price with no caching."""
    return TOKENS_IN * model["prompt"] + TOKENS_OUT * model["completion"]


models = fetch_models()
tool_models = [model for model in models if "tools" in model["supported_parameters"]]
paid = sorted(
    (model for model in tool_models
     if model["prompt"] > 0 and model["completion"] >= 0 and ":batch" not in model["id"] and not model["id"].startswith("~")),
    key=per_claim,
)
FREE_MODELS = ["openrouter/free", "nvidia/nemotron-3-super-120b-a12b:free", "google/gemma-4-31b-it:free"]
available = {model["id"] for model in tool_models}
assert all(model_id in available for model_id in FREE_MODELS), [model_id for model_id in FREE_MODELS if model_id not in available]

print(f"{len(models)} models, {len(tool_models)} support tools. Full agent on Luna: {TOKENS_IN:,.0f} input and {TOKENS_OUT:,.0f} output tokens a claim")
pd.DataFrame([
    {"model": model["id"], "input $/M": model["prompt"] * 1e6, "output $/M": model["completion"] * 1e6, "estimated cost per claim": per_claim(model)}
    for model in paid[:10]
]).style.format({"input $/M": "{:.3f}", "output $/M": "{:.3f}", "estimated cost per claim": "${:.5f}"})

In [ ]:
def slug(model_id):
    """A file-safe name for a model."""
    return re.sub(r"[^a-z0-9]+", "_", model_id.lower()).strip("_")


def full_agent_on(model_id):
    """The v2 full agent on another model, at low reasoning where the model takes a reasoning effort."""
    params = next(model["supported_parameters"] for model in tool_models if model["id"] == model_id)
    return Config(f"full_{slug(model_id)}", model_id, effort="low" if "reasoning_effort" in params else None,
                  manual_and_history=True, payout_tool=True)


SWEEP = [full_agent_on(model["id"]) for model in paid[:5]] + [full_agent_on(model_id) for model_id in FREE_MODELS]
FREE_NAMES = {full_agent_on(model_id).name for model_id in FREE_MODELS}
FREE_WORKERS = 2  # Free models allow about 20 requests a minute

sweep_pilot = {
    config.name: run_config(config, 1, pilot_claims, labels, set_name="pilot", workers=FREE_WORKERS if config.name in FREE_NAMES else WORKERS)
    for config in SWEEP
}
pd.DataFrame([
    {"configuration": name, "right": sum(row["right"] for row in rows), "cost per claim": sum(row["cost"] for row in rows) / len(rows),
     "estimated three runs": sum(row["cost"] for row in rows) / len(rows) * 75, "stopped or failed": sum(bool(row["error"] or row["stopped"]) for row in rows)}
    for name, rows in sweep_pilot.items()
]).set_index("configuration").style.format({"cost per claim": "${:.5f}", "estimated three runs": "${:.4f}"})

In [ ]:
sweep = {
    config.name: run_repeats(config, claims, labels, cutoff=10, workers=FREE_WORKERS if config.name in FREE_NAMES else WORKERS)
    for config in SWEEP
}
sweep_and_luna = {"C_full_v2": ablation_v2["C_full_v2"]} | sweep
results_table(sweep_and_luna)

In [ ]:
miss_table(sweep_and_luna)

Two kinds of zero in this table are not about a model's judgment:

- **The free models were rate-limited, not measured.** Almost all of their claims failed with OpenRouter's `429` rate-limit error (the "stopped or failed" column). `openrouter/free` got 12 of 25 on its first run before the limit hit. Their scores say nothing about the models, and they would need a day each, or an account with credits, to measure.
- **`gpt-oss-20b` and `mistral-nemo` mostly answered in plain text** instead of calling `record_decision`, so most of their claims have nothing recorded. On this harness they can't do the job, at any price.

## Headline: same model, better tools

The cheapest configuration that meets the target (at least 15 of 25 in *every* run) picks the model. Ed's original agent then runs on that same model, 3 times, beside the full agent.

In [ ]:
def meets_target(runs, target=15, needed_runs=3):
    """True when a configuration ran every repeat and got at least target right in each."""
    return len(runs) >= needed_runs and min(sum(row["right"] for row in run) for run in runs) >= target


candidates = [summarize(name, runs) | {"name": name} for name, runs in sweep_and_luna.items() if meets_target(runs)]
cheapest = min(candidates, key=lambda summary: (summary["cost per run"], -summary["mean"]))
chosen_config = next(config for config in [ABLATION_V2[-1], *SWEEP] if config.name == cheapest["name"])
print(f"Cheapest configuration that meets the target: {chosen_config.name} ({chosen_config.model}), "
      f"min {cheapest['min']}, mean {cheapest['mean']}, ${cheapest['cost per run']:.4f} a run")

if chosen_config.model == LUNA:
    ed_name, ed_runs = ED_ORIGINAL.name, ablation_v1[ED_ORIGINAL.name]
else:
    ed_on_chosen = Config(f"ed_original_on_{slug(chosen_config.model)}", chosen_config.model, effort=chosen_config.effort, ed_original=True)
    workers = FREE_WORKERS if chosen_config.name in FREE_NAMES else WORKERS
    ed_name, ed_runs = ed_on_chosen.name, run_repeats(ed_on_chosen, claims, labels, workers=workers)
headline = {ed_name: ed_runs, chosen_config.name: sweep_and_luna[chosen_config.name]}
results_table(headline)

In [ ]:
miss_table(headline)

In [ ]:
right_by_claim(headline, labels)

## Held-out set

The 25 held-out claims were never looked at while the agent was built or tuned. These three configurations were chosen before any held-out claim was run, and nothing changed afterwards: the same code and the same prompts as on the queue, 3 runs each.

1. The full agent on `ling-3.0-flash`, the cheapest model that met the target on the queue
2. Ed's original agent on `ling-3.0-flash`, for the same-model comparison
3. The full agent on `deepseek-v4-flash-0731`, the most accurate on the queue

A run is skipped if it could take the total spend past $0.50, judging by what the configuration cost a run on the queue.

In [ ]:
LING = "inclusionai/ling-3.0-flash"
DEEPSEEK = "deepseek/deepseek-v4-flash-0731"
HELD_OUT_CAP = 0.50

held_claims, held_labels = get_sample_data(held_out=True)
ling_full = full_agent_on(LING)
HELD_OUT = [
    ling_full,
    Config(f"ed_original_on_{slug(LING)}", LING, effort=ling_full.effort, ed_original=True),
    full_agent_on(DEEPSEEK),
]
queue_cost = {name: summarize(name, runs)["cost per run"] for name, runs in (sweep_and_luna | headline).items()}

held_out, capped = {}, False
for config in HELD_OUT:
    held_out[config.name] = []
    for run in range(1, 4):
        cached = (RESULTS / "held_out" / f"{config.name}__run{run}.json").exists()
        if not cached and (capped or spent() + queue_cost[config.name] > HELD_OUT_CAP):
            capped = True
            print(f"Skipped {config.name} run {run}: it could take the spend past ${HELD_OUT_CAP:.2f}")
            continue
        held_out[config.name].append(run_config(config, run, held_claims, held_labels, set_name="held_out"))
held_out = {name: runs for name, runs in held_out.items() if runs}
results_table(held_out)

In [ ]:
miss_table(held_out)

In [ ]:
right_by_claim(held_out, held_labels)

The full agent's misses on the held-out set, on `ling-3.0-flash`:

In [ ]:
show_misses(held_out[ling_full.name], held_labels)

## Free models, retested

The sweep's free models were rate-limited, not measured. This retest runs the full agent on three free models with one claim at a time, at most 12 model calls a minute, and a backoff on a 429: a claim waits 30, 60, 120 and then 240 seconds before it gives up. A model whose claim is still rate-limited after that stops, and is reported as rate-limited rather than scored. Everything stops cleanly at 07:00.

The rules were chosen before the retest started. A model with fewer than 10 right on run 1 stops there. If any free model gets at least 15 in all 3 runs, the one with the highest mean (then the fastest) gets 3 held-out runs with the same settings, and becomes the cheapest answer at $0.

In [ ]:
RETEST_DEADLINE = datetime(2026, 10, 19, 7, 0)
RETEST_MODELS = ["openrouter/free", "nvidia/nemotron-3.5-lightning:free", "google/gemma-4-31b-it:free"]


def paced(config):
    """A free model's configuration, paced and retried for the per-minute limit."""
    return replace(config, name=f"{config.name}_retest", requests_per_minute=12, rate_limit_retries=4)


FREE_RETEST = [paced(full_agent_on(model_id)) for model_id in RETEST_MODELS]
retest, retest_status = {}, {}
for config in FREE_RETEST:
    try:
        retest[config.name] = run_repeats(config, claims, labels, cutoff=10, workers=1, deadline=RETEST_DEADLINE)
        retest_status[config.name] = "complete" if len(retest[config.name]) == 3 else "dropped after run 1"
    except RunAborted as stopped:
        print(stopped)
        retest[config.name] = cached_runs(config, labels)
        retest_status[config.name] = str(stopped).split(": ", 1)[1]
retest = {name: runs for name, runs in retest.items() if runs}
print(retest_status)
results_table(retest) if retest else "No retest run finished"

In [ ]:
free_winners = [summarize(name, runs) | {"name": name} for name, runs in retest.items() if meets_target(runs)]
held_out_free = {}
if free_winners:
    best_free = max(free_winners, key=lambda summary: (summary["mean"], -summary["seconds per claim"]))
    free_config = next(config for config in FREE_RETEST if config.name == best_free["name"])
    print(f"{free_config.name} meets the target at $0: min {best_free['min']}, mean {best_free['mean']}. Running it on the held-out set.")
    try:
        held_out_free[free_config.name] = run_repeats(free_config, held_claims, held_labels, set_name="held_out", workers=1, deadline=RETEST_DEADLINE)
    except RunAborted as stopped:
        print(stopped)
        held_out_free[free_config.name] = cached_runs(free_config, held_labels, set_name="held_out")
        retest_status[f"{free_config.name} held-out"] = str(stopped).split(": ", 1)[1]
    held_out_free = {name: runs for name, runs in held_out_free.items() if runs}
else:
    print("No free model met the target of at least 15 in every run.")
results_table(held_out_free) if held_out_free else None

## Results table and chart

`results.csv` holds one row for each configuration on each set. The chart plots mean correct against cost per run for every configuration with all 3 runs on the queue (the v2 ablation, Ed's agent and the sweep), with min to max as error bars. The line links the configurations that are best for their price: none of the others is both cheaper and more accurate. The hollow markers are the same configurations on the held-out set.

In [ ]:
ALL_CONFIGS = {config.name: config for config in ABLATION_V1 + ABLATION_V2 + SWEEP + HELD_OUT + FREE_RETEST}


def note_for(name, runs):
    """Why a configuration's score is not a measure of the model, if it isn't."""
    rows = [row for run in runs for row in run]
    if name in FREE_NAMES:
        return "rate-limited (429), untested"
    if retest_status.get(name, "complete") not in ("complete", "dropped after run 1"):
        return f"retest {retest_status[name]}"
    if sum(row["decision"] == "nothing recorded" for row in rows) > len(rows) / 2:
        return "mostly answered without calling record_decision"
    return ""


def results_frame():
    """One row for each configuration on each set."""
    sets = {"queue": ablation_v1 | ablation_v2 | sweep_and_luna | headline | retest, "held_out": held_out | held_out_free}
    rows = []
    for set_name, experiments in sets.items():
        for name, runs in experiments.items():
            config = ALL_CONFIGS[name]
            rows.append({"set": set_name, "model": config.model, "version": config.version, **summarize(name, runs), "note": note_for(name, runs)})
    return pd.DataFrame(rows)


results = results_frame()
results.to_csv("results.csv", index=False, float_format="%.6g")
results[["set", "configuration", "model", "min", "mean", "max", "cost per run", "cost per correct", "seconds per claim", "calls per claim", "note"]]

In [ ]:
import plotly.graph_objects as go  # noqa: E402

BLUE, ORANGE, INK, MUTED, GRID, SURFACE = "#2a78d6", "#eb6834", "#1f1f1e", "#6b6a64", "#e6e5e0", "#fcfcfb"


MODEL_NAMES = {LUNA: "Luna", LING: "Ling", DEEPSEEK: "DeepSeek", "qwen/qwen3.7-flash": "Qwen"}
# Where each label sits, so that neighbouring labels don't collide; the rest sit to the right of their marker
LABEL_POSITIONS = {
    ("queue", "ed_original"): "bottom right",
    ("queue", f"ed_original_on_{slug(LING)}"): "bottom right",
    ("held_out", f"ed_original_on_{slug(LING)}"): "bottom left",
    ("queue", f"full_{slug(LING)}"): "top left",
    ("queue", f"full_{slug(DEEPSEEK)}"): "top left",
    ("held_out", f"full_{slug(DEEPSEEK)}"): "bottom center",
    ("held_out", f"full_{slug(LING)}"): "top center",
}


def label(name):
    """A short chart label for a configuration."""
    config = ALL_CONFIGS[name]
    model = MODEL_NAMES.get(config.model, config.model.split("/")[-1])
    kind = {"A_lean_v2": "Lean", "B_manual_history_v2": "Manual + history"}.get(name, "Ed's agent" if config.ed_original else "Full agent")
    return f"{kind}, {model}"


def best_for_price(table):
    """The configurations that no other beats on both cost and accuracy, cheapest first."""
    best, keep = -math.inf, []
    for _, row in table.sort_values(["cost per run", "mean"], ascending=[True, False]).iterrows():
        if row["mean"] > best:
            keep.append(row)
            best = row["mean"]
    return pd.DataFrame(keep)


def scatter(frame, color, symbol, name, filled):
    """One set of configurations as labelled markers with min to max error bars."""
    set_name = frame["set"].iloc[0]
    suffix = " (held-out)" if set_name == "held_out" else ""
    return go.Scatter(
        x=frame["cost per run"], y=frame["mean"], mode="markers+text", name=name,
        marker={"color": color, "size": 10, "symbol": symbol, "line": {"color": SURFACE if filled else color, "width": 2}},
        error_y={"type": "data", "symmetric": False, "array": frame["max"] - frame["mean"], "arrayminus": frame["mean"] - frame["min"],
                 "color": color, "thickness": 1.5, "width": 4},
        text=[label(config) + suffix for config in frame["configuration"]],
        textposition=[LABEL_POSITIONS.get((set_name, config), "middle right") for config in frame["configuration"]],
        textfont={"color": INK, "size": 11},
        customdata=frame[["configuration", "min", "max"]],
        hovertemplate="%{customdata[0]}<br>mean %{y} (min %{customdata[1]}, max %{customdata[2]})<br>$%{x:.4f} a run<extra></extra>",
    )


# A $0 configuration can't sit on a log axis, so free models appear in the table only
queue = results[(results["set"] == "queue") & (results["runs"] == 3) & (results["note"] == "") & (results["version"] == 2) & (results["cost per run"] > 0)]
frontier = best_for_price(queue)
held = results[(results["set"] == "held_out") & (results["cost per run"] > 0)]

fig = go.Figure()
fig.add_hline(y=15, line={"color": MUTED, "width": 1, "dash": "dot"}, annotation_text="target: 15 of 25",
              annotation_position="bottom right", annotation_font={"color": MUTED, "size": 11})
fig.add_trace(go.Scatter(x=frontier["cost per run"], y=frontier["mean"], mode="lines", line={"color": BLUE, "width": 2},
                         name="Best for the price (queue)", hoverinfo="skip"))
fig.add_trace(scatter(queue, BLUE, "circle", "Queue: mean, min to max", filled=True))
fig.add_trace(scatter(held, ORANGE, "diamond-open", "Held-out set", filled=False))

# Arrows from Ed's agent to the full agent on the same model, on the queue and on the held-out set
ED_LING, FULL_LING = f"ed_original_on_{slug(LING)}", f"full_{slug(LING)}"
for frame, color in [(queue, BLUE), (held, ORANGE)]:
    points = frame.set_index("configuration")
    if {ED_LING, FULL_LING} <= set(points.index):
        start, end = points.loc[ED_LING], points.loc[FULL_LING]
        fig.add_annotation(
            x=math.log10(end["cost per run"]), y=end["mean"], ax=math.log10(start["cost per run"]), ay=start["mean"],
            xref="x", yref="y", axref="x", ayref="y", showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=1.5,
            arrowcolor=color, standoff=7, startstandoff=7, text="",
        )

held_points = held.set_index("configuration")
ed_held, full_held = held_points.loc[ED_LING], held_points.loc[FULL_LING]
price = "for less per run" if full_held["cost per run"] < ed_held["cost per run"] else "for about the same per run"
TITLE = (f"Same cheap model, better tools: Ed's agent {ed_held['mean']:.1f} vs mine {full_held['mean']:.1f} on unseen claims, {price}"
         f"<br><sup>Claims right out of 25 against cost per run, 3 runs each, on {MODEL_NAMES[LING]} and the other models tested</sup>")
fig.update_layout(
    title={"text": TITLE, "font": {"color": INK, "size": 15}},
    xaxis={"title": "Cost per run of 25 claims (US dollars, log scale)", "type": "log", "gridcolor": GRID, "color": MUTED,
           "range": [math.log10(0.0025), math.log10(0.05)], "tickvals": [0.003, 0.005, 0.01, 0.02, 0.03],
           "ticktext": ["$0.003", "$0.005", "$0.01", "$0.02", "$0.03"]},
    yaxis={"title": "Claims right (mean, with min to max)", "range": [10, 23], "gridcolor": GRID, "color": MUTED},
    plot_bgcolor=SURFACE, paper_bgcolor=SURFACE, font={"color": INK}, width=900, height=560,
    legend={"orientation": "h", "y": -0.18, "x": 0}, margin={"r": 40},
)
try:
    fig.write_image("results.png", scale=2)
except (ValueError, ImportError, RuntimeError) as error:
    print(f"Could not write results.png ({error}). Writing it needs kaleido, for example: uv run --with kaleido jupyter lab")
fig.show()

In [ ]:
print(f"Total spent on every cached run: ${spent():.4f}")